# FalkorDBVectorStore
<a href="https://docs.falkordb.com/" target="_blank">FalkorDB</a> 是一个开源图数据库，集成了向量相似性搜索功能

它支持：
- 近似最近邻搜索
- 欧氏相似度与余弦相似度
- 结合向量和关键词搜索的混合搜索

本 Notebook 展示了如何使用 FalkorDB 向量索引 (`FalkorDB`)

请参阅 <a href="https://docs.falkordb.com/" target="_blank">安装说明</a>

## 设置

In [ ]:
# Pip install necessary package
%pip install --upgrade  falkordb
%pip install --upgrade  tiktoken
%pip install --upgrade  langchain langchain_huggingface


Note: you may need to restart the kernel to use updated packages.



### 凭证
我们要使用 `HuggingFace`，因此需要获取 HuggingFace API 密钥。

In [1]:
import getpass
import os

if "HUGGINGFACE_API_KEY" not in os.environ:
    os.environ["HUGGINGFACE_API_KEY"] = getpass.getpass("HuggingFace API Key:")

如果您想获得模型调用的自动化跟踪，您也可以通过取消注释下方的内容来设置您的 LangSmith API 密钥：

In [2]:
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")
# os.environ["LANGSMITH_TRACING"] = "true"

## 初始化

In [3]:
from langchain_community.vectorstores.falkordb_vector import FalkorDBVector
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

您可以在本地使用 FalkorDBVector 和 Docker。请参阅<a href="https://docs.falkordb.com/" target="_blank">安装说明</a>

In [4]:
host = "localhost"
port = 6379

或者您可以使用 FalkorDBVector 配合 <a href="https://app.falkordb.cloud">FalkorDB Cloud</a>

In [5]:
# E.g
# host = "r-6jissuruar.instance-zwb082gpf.hc-v8noonp0c.europe-west1.gcp.f2e0a955bb84.cloud"
# port = 62471
# username = "falkordb" # SET ON FALKORDB CLOUD
# password = "password" # SET ON FALKORDB CLOUD

In [6]:
vector_store = FalkorDBVector(host=host, port=port, embedding=HuggingFaceEmbeddings())

## 管理向量存储

This guide explains how to manage your vector stores.
本指南将介绍如何管理您的向量存储。

A vector store is a database that stores vector embeddings.
向量存储是存储向量嵌入的数据库。

Vector embeddings are numerical representations of data, such as text or images.
向量嵌入是数据的数值表示，例如文本或图像。

They are used in machine learning tasks like similarity search, recommendation systems, and natural language processing.
它们用于机器学习任务，如相似性搜索、推荐系统和自然语言处理。

### Creating a vector store
### 创建向量存储

You can create a vector store using the following code:
您可以使用以下代码创建向量存储：

```python
from vectorstores import VectorStore

# Create a new vector store
vector_store = VectorStore.create("my_vector_store")
```

This will create a new vector store named "my_vector_store".
这将创建一个名为“my_vector_store”的新向量存储。

### Adding data to a vector store
### 向向量存储添加数据

You can add data to a vector store using the `add` method:
您可以使用 `add` 方法向向量存储添加数据：

```python
# Add data to the vector store
vector_store.add(data=["This is a document.", "This is another document."])
```

This will add two documents to the vector store.
这将向向量存储添加两个文档。

### Searching a vector store
### 搜索向量存储

You can search a vector store using the `search` method:
您可以使用 `search` 方法搜索向量存储：

```python
# Search the vector store for similar documents
results = vector_store.search(query="What is a vector store?")
```

This will return a list of documents that are similar to the query.
这将返回与查询相似的文档列表。

### Deleting a vector store
### 删除向量存储

You can delete a vector store using the `delete` method:
您可以使用 `delete` 方法删除向量存储：

```python
# Delete the vector store
vector_store.delete()
```

This will delete the vector store and all of its data.
这将删除向量存储及其所有数据。

### Managing vector store metadata
### 管理向量存储元数据

You can also manage metadata associated with your vector store.
您还可以管理与向量存储关联的元数据。

Metadata can include information such as the source of the data, the date it was added, or any other relevant details.
元数据可以包含诸如数据来源、添加日期或任何其他相关详细信息之类的信息。

You can add metadata when creating a vector store or by updating existing entries.
您可以在创建向量存储时添加元数据，或通过更新现有条目来添加元数据。

```python
# Add data with metadata
vector_store.add(data=["This is a document."], metadata={"source": "web", "date": "2023-10-27"})

# Update metadata for an existing entry
vector_store.update_metadata(id="doc1", metadata={"source": "local", "date": "2023-10-28"})
```

This allows for more organized and searchable data within your vector store.
这使得在向量存储中的数据更加有组织且易于搜索。

### 向向量存储添加项目

In [7]:
from langchain_core.documents import Document

document_1 = Document(page_content="foo", metadata={"source": "https://example.com"})

document_2 = Document(page_content="bar", metadata={"source": "https://example.com"})

document_3 = Document(page_content="baz", metadata={"source": "https://example.com"})

documents = [document_1, document_2, document_3]

vector_store.add_documents(documents=documents, ids=["1", "2", "3"])

['1', '2', '3']

### 更新向量存储中的项目

In [8]:
updated_document = Document(
    page_content="qux", metadata={"source": "https://another-example.com"}
)

vector_store.update_documents(document_id="1", document=updated_document)

### 从向量存储中删除项目

In [9]:
vector_store.delete(ids=["3"])

## 查询向量存储

一旦您的向量存储创建完成并且已添加了相关文档，您很可能希望在链或代理运行时查询它。

### 直接查询

执行简单的相似性搜索可以按以下方式进行：

In [10]:
results = vector_store.similarity_search(
    query="thud", k=1, filter={"source": "https://another-example.com"}
)
for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")

* qux [{'text': 'qux', 'id': '1', 'source': 'https://another-example.com'}]


如果您想执行相似性搜索并获取相应的得分，可以运行：

In [11]:
results = vector_store.similarity_search_with_score(query="bar")
for doc, score in results:
    print(f"* [SIM={score:3f}] {doc.page_content} [{doc.metadata}]")

* [SIM=0.000001] bar [{'text': 'bar', 'id': '2', 'source': 'https://example.com'}]


### 将查询转换为检索器

您也可以将向量存储转换为检索器，以便在链中使用，从而更方便。

In [12]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 1})
retriever.invoke("thud")

[Document(metadata={'text': 'qux', 'id': '1', 'source': 'https://another-example.com'}, page_content='qux')]

## 用于检索增强生成的使用方法
有关如何将此向量存储用于检索增强生成（RAG）的指南，请参阅以下章节：
- <a href="https://python.langchain.com/v0.2/docs/tutorials/#working-with-external-knowledge" target="_blank">教程：处理外部知识</a>
- <a href="https://python.langchain.com/v0.2/docs/how_to/#qa-with-rag" target="_blank">操作指南：使用RAG进行问答</a>
- <a href="Retrieval conceptual docs" target="_blank">检索概念文档</a>

## API 参考
有关 `FalkorDBVector` 所有功能和配置的详细文档，请访问 API 参考：https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.falkordb_vector.FalkorDBVector.html